In [ ]:
import numpy as np
import cupy as cp
import pickle

from cuml.svm import SVC
from cuml.preprocessing import StandardScaler
from cuml.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
latent_dim = 128

X = np.fromfile("output/train_features.bin", dtype=np.float32)
y = np.fromfile("output/train_labels.bin", dtype=np.uint16)

N = y.shape[0]
X = X.reshape(N, latent_dim)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
X_gpu = cp.asarray(X)
y_gpu = cp.asarray(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_gpu, y_gpu, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [ ]:
svm = SVC(
    kernel="rbf",
    C=10,
    gamma="scale"
)

svm.fit(X_train, y_train)

In [ ]:
y_pred = svm.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("SVM Accuracy:", float(acc))

In [ ]:
with open("svm_cuml.pkl", "wb") as f:
    pickle.dump(svm, f)